In [ ]:
import datetime
from tqdm import tqdm
import os
import fnmatch
import pickle

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data import Dataset as TorchDataset
import torch.nn.functional as F
from torchvision import datasets, transforms
from typing import Tuple, List, Type, Dict, Any
from torch.utils.tensorboard import SummaryWriter

from SGDR import CosineAnnealingWarmRestarts
from MyDataPreparationSeq import CustomDataset
from TransformerMask1d import SequenceToVectorTransformer

device = torch.device('cuda:1')

In [ ]:
from wind_Transformer1D_002 import mask_one_timestep

In [ ]:
dataset = CustomDataset(data_path="/app/Kara_plume_movement/extracted_features/extracted_wind_007_6hrs", years=[year for year in range(1979, 2024+1)])

In [ ]:
batch_size = 16
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
run_name = 'wind_Transformer1D_run002'
model = torch.load(f'/app/Kara_plume_movement/wind/models/model_{run_name}.pth', map_location=torch.device('cpu'));
model.eval();
model = model.cuda()

In [ ]:
data = next(iter(dataloader))
data.shape

In [ ]:
loss_function=torch.nn.MSELoss()

In [ ]:
mask_value = 0.0

In [ ]:
test_loss = 0

with torch.no_grad():
    for batch_data in dataloader:
        x = batch_data.to(device='cuda', dtype=torch.float)  # [N, T, D]

        # Маскирование через mask_one_timestep
        x_masked, target, t_idx = mask_one_timestep(x, mask_value=mask_value)

        # Forward
        pred = model.forward(x_masked)  # [N, D]

        # Loss
        loss = loss_function(pred, target)
        test_loss += loss.detach() * batch_size

In [ ]:
test_loss

In [ ]:
pred.shape, target.shape

In [ ]:
pred.mean(), pred.std(), target.mean(), target.std()